In [1]:
# ============================================================
# Cell 1 — Imports and paths
# ============================================================

from pathlib import Path

import rasterio
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "gee_exports"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

Raw: c:\Projects\floodske\floodske\data\raw\gee_exports
Processed: c:\Projects\floodske\floodske\data\processed


In [2]:
# ============================================================
# Cell 2 — Define input raster paths
# ============================================================

raster_paths = {
    "dem": RAW_DIR / "kenya_dem_srtm_90m.tif",
    "slope": RAW_DIR / "kenya_slope_srtm_90m.tif",
    "rainfall": RAW_DIR / "kenya_chirps_rainfall_2000_2025_90m.tif",
    "landcover": RAW_DIR / "kenya_esa_worldcover_2021_90m.tif",
    "distance_to_water": RAW_DIR / "kenya_distance_to_water_90m.tif",
}

for name, path in raster_paths.items():
    print(name, "exists:", path.exists(), "|", path)

dem exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_dem_srtm_90m.tif
slope exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_slope_srtm_90m.tif
rainfall exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_chirps_rainfall_2000_2025_90m.tif
landcover exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_esa_worldcover_2021_90m.tif
distance_to_water exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_distance_to_water_90m.tif


In [3]:
# ============================================================
# Cell 3 — Validate raster metadata
# ============================================================

for name, path in raster_paths.items():
    print("\n" + "=" * 60)
    print(name.upper())

    with rasterio.open(path) as src:
        print("CRS:", src.crs)
        print("Width x Height:", src.width, "x", src.height)
        print("Bounds:", src.bounds)
        print("Resolution:", src.res)
        print("Band count:", src.count)
        print("NoData:", src.nodata)
        print("Dtype:", src.dtypes[0])


DEM
CRS: EPSG:4326
Width x Height: 9890 x 12482
Bounds: BoundingBox(left=33.910234165642585, bottom=-4.677078526768288, right=41.90613850959045, top=5.414415711973592)
Resolution: (0.0008084837557075694, 0.0008084837557075694)
Band count: 1
NoData: None
Dtype: int16

SLOPE
CRS: EPSG:4326
Width x Height: 9890 x 12482
Bounds: BoundingBox(left=33.910234165642585, bottom=-4.677078526768288, right=41.90613850959045, top=5.414415711973592)
Resolution: (0.0008084837557075694, 0.0008084837557075694)
Band count: 1
NoData: None
Dtype: float32

RAINFALL
CRS: EPSG:4326
Width x Height: 9890 x 12482
Bounds: BoundingBox(left=33.910234165642585, bottom=-4.677078526768288, right=41.90613850959045, top=5.414415711973592)
Resolution: (0.0008084837557075694, 0.0008084837557075694)
Band count: 1
NoData: None
Dtype: float64

LANDCOVER
CRS: EPSG:4326
Width x Height: 9890 x 12482
Bounds: BoundingBox(left=33.910234165642585, bottom=-4.677078526768288, right=41.90613850959045, top=5.414415711973592)
Resolution

In [4]:
# ============================================================
# Cell 4 — Inspect raster value ranges
# ============================================================

for name, path in raster_paths.items():
    print("\n" + "=" * 60)
    print(name.upper())

    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")

        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        print("Min:", np.nanmin(arr))
        print("Max:", np.nanmax(arr))
        print("Mean:", np.nanmean(arr))
        print("NaN cells:", np.isnan(arr).sum())


DEM
Min: -12.0
Max: 5030.0
Mean: 467.91354
NaN cells: 0

SLOPE
Min: 0.0
Max: 77.88144
Mean: 3.1766622
NaN cells: 49873490

RAINFALL
Min: 512.94556
Max: 63506.438
Mean: 16474.715
NaN cells: 49873962

LANDCOVER
Min: 0.0
Max: 95.0
Mean: 15.701256
NaN cells: 0

DISTANCE_TO_WATER
Min: 0.0
Max: 89646.375
Mean: 21582.887
NaN cells: 49830317


In [ ]:
# ============================================================
# Cell 5 — Check valid pixel coverage
# ============================================================

for name, path in raster_paths.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")

        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        total_pixels = arr.size
        valid_pixels = np.count_nonzero(~np.isnan(arr))
        pct_valid = (valid_pixels / total_pixels) * 100

        print(
            f"{name:20s} "
            f"valid={valid_pixels:,} "
            f"total={total_pixels:,} "
            f"({pct_valid:.2f}%)"
        )